# WISER Module 4: Making Meaning with NLP Semantics

**Learning goal:** Explore synthetic text, compare note meanings, examine changing terminology, and explain the limits of automated matches.

**Before you begin**
1. Open this file in Google Colab and select **File → Save a copy in Drive**.
2. Select **Runtime → Run all** once. If Colab asks to connect to a runtime, choose **Connect** and run again.
3. Read the prompts, edit the response cells, and rerun the cells you change.
4. At the end, download the ZIP file and your completed notebook (File → Download → Download .ipynb).

**Privacy:** Use the WISER synthetic data. Do not upload identifiable clinical notes, protected health information, or unapproved institutional data.

**What to submit:** Your completed notebook and the ZIP file created at the end. Your instructor may specify a different submission location.

In [ ]:
MODULE_ID = "Module 4"
NOTEBOOK_TITLE = "WISER Module 4: NLP Semantics"
from pathlib import Path
import json
import re
import shutil
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Allows a local Python run outside Colab.

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

STIGMA_TERMS = {
    "addict": "person with a substance use disorder",
    "abuser": "person with a substance use disorder",
    "drug abuse": "substance use or substance use disorder",
    "substance abuse": "substance use or substance use disorder",
    "clean": "describe the specific result or situation",
    "dirty": "describe the specific toxicology result",
    "junkie": "person with a substance use disorder",
}

def stigma_audit(text):
    """Suggest terms to review; context and human judgment remain essential."""
    s = str(text).lower()
    return {term: replacement for term, replacement in STIGMA_TERMS.items()
            if re.search(r"(?<!\w)" + re.escape(term) + r"(?!\w)", s)}

def quick_profile(df):
    print(f"Rows: {len(df):,}; columns: {len(df.columns)}")
    display(pd.DataFrame({"type": df.dtypes.astype(str),
                          "missing": df.isna().sum(),
                          "unique": df.nunique(dropna=True)}))
    display(df.head(5))
print("Setup complete. Run the next cell to load data.")

Setup complete. Run the next cell to load data.


## 1. Load and inspect the notes

The notebook tries to load `notes.csv` from the WISER GitHub repository. If Colab cannot reach it, a small synthetic practice dataset loads and the notebook says so. **Check the printed data source before submitting:** if you used practice data, ask your instructor whether that is acceptable.

The text is synthetic, but always treat real clinical text as sensitive.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mcbacloudlab/WISERTraining-Modules-/main/data/notes.csv"
practice_notes = pd.DataFrame({
    "note_id": range(1, 9),
    "year": [2014, 2015, 2017, 2018, 2020, 2021, 2022, 2024],
    "setting": ["clinic", "survey", "clinic", "public_health", "clinic", "survey", "clinic", "public_health"],
    "text": [
        "Patient reports nonmedical opioid use and limited access to MOUD.",
        "Respondent endorses pain reliever misuse in prior year.",
        "Person with OUD reports transportation barriers and unstable housing.",
        "Heroin use variable renamed in surveillance extract; definition changed from prior wave.",
        "Patient denies opioid use but reports cravings and missed appointments.",
        "Participant reports fentanyl exposure and interest in treatment.",
        "Person receiving buprenorphine reports improved functioning.",
        "Pain reliever dependence field crosswalked to DSM-5 OUD severity proxy."
    ],
    "substance_class": ["opioid", "opioid", "opioid", "opioid", "opioid", "synthetic_opioid", "opioid", "opioid"]
})
try:
    notes = pd.read_csv(DATA_URL)
    data_source = "WISER repository: notes.csv"
except Exception as error:
    notes = practice_notes.copy()
    data_source = "8-row synthetic practice dataset"
    print("Could not load the repository CSV:", type(error).__name__, str(error)[:180])
    print("Using the small synthetic practice dataset. Check with your instructor before submitting.")

required = {"note_id", "text"}
missing = required - set(notes.columns)
if missing:
    raise ValueError(f"The loaded file is missing required columns: {sorted(missing)}. Check the CSV and DATA_URL.")
if notes["note_id"].isna().any() or notes["note_id"].duplicated().any():
    raise ValueError("note_id must be unique and nonmissing for the similarity table.")
notes["text"] = notes["text"].fillna("").astype(str)
if not notes["text"].str.strip().any():
    raise ValueError("No note text was found in the loaded file.")
print("Data source:", data_source)
quick_profile(notes)
print("Terms to review (first 5 notes):")
display(notes[["note_id", "text"]].head(5).assign(
    wording_to_review=notes["text"].head(5).map(stigma_audit)))

Could not load the repository CSV: HTTPError HTTP Error 404: Not Found
Using the small synthetic practice dataset. Check with your instructor before submitting.
Data source: 8-row synthetic practice dataset
Rows: 8; columns: 5


,type,missing,unique
note_id,int64,0,8
year,int64,0,8
setting,object,0,3
text,object,0,8
substance_class,object,0,2


,note_id,year,setting,text,substance_class
0,1,2014,clinic,Patient reports nonmedical opioid use and limi...,opioid
1,2,2015,survey,Respondent endorses pain reliever misuse in pr...,opioid
2,3,2017,clinic,Person with OUD reports transportation barrier...,opioid
3,4,2018,public_health,Heroin use variable renamed in surveillance ex...,opioid
4,5,2020,clinic,Patient denies opioid use but reports cravings...,opioid


Terms to review (first 5 notes):


,note_id,text,wording_to_review
0,1,Patient reports nonmedical opioid use and limi...,{}
1,2,Respondent endorses pain reliever misuse in pr...,{}
2,3,Person with OUD reports transportation barrier...,{}
3,4,Heroin use variable renamed in surveillance ex...,{}
4,5,Patient denies opioid use but reports cravings...,{}


### Your response: language and context
Which text fields might reflect patient language, clinician language, or survey wording? What **additional information** would you need before deciding? Explain why the source of the language matters.

Type between the brackets below. The notebook will save your response in the final ZIP.

In [ ]:
response_language = """[Write 2–4 sentences here.]"""
print(response_language)

## 2. Prepare text for comparison

This simple function converts letters to lowercase and removes punctuation. Compare the original and cleaned text below. Removing punctuation can also remove meaning; later, consider whether a model should instead use the original text.

In [ ]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

notes["text_clean"] = notes["text"].map(normalize_text)
display(notes[["note_id", "text", "text_clean"]].head(8))

## 3. Compare note meanings

The cell first attempts a pretrained sentence embedding model. A model download may take a minute. If it is unavailable, the notebook uses TF-IDF (word overlap) and clearly reports that change. Both methods produce **similarity scores**, not a clinical diagnosis. Scores closer to 1 mean more similar under the method used.

Look for two notes that seem similar and one pair whose score is surprising. Would your interpretation change if the method switched?

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

model = None
vectorizer = None
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    X = model.encode(notes["text_clean"].tolist())
    method = "sentence embeddings (all-MiniLM-L6-v2)"
except Exception as error:
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    X = vectorizer.fit_transform(notes["text_clean"])
    method = "TF-IDF (word overlap)"
    print("Embedding model unavailable:", type(error).__name__, str(error)[:180])
if X.shape[1] == 0:
    raise ValueError("The notes contain no usable terms after cleaning.")
sim = cosine_similarity(X)
sim_df = pd.DataFrame(sim, index=notes["note_id"], columns=notes["note_id"])
print("Method:", method)
display(sim_df.round(2).iloc[:10, :10])
print("Example: the diagonal scores are 1 because each note is compared with itself.")

In [ ]:
response_similarity = """[Identify a pair of note IDs, describe its score, and explain whether the match makes sense. Mention one limitation.]"""
print(response_similarity)

## 4. Terminology crosswalk

A crosswalk documents how terms differ across sources or years. The rows below are **examples for review**, not automatic clinical equivalences. In particular, a survey measure of “misuse” cannot automatically be relabeled as a diagnosis. Review the definitions and time periods before combining data.

**Your task:** Edit or add at least one row in the table and explain the evidence you would need to approve that mapping.

In [ ]:
crosswalk = pd.DataFrame({
    "legacy_term": ["pain reliever dependence", "opioid misuse", "heroin use", "substance abuse"],
    "candidate_term": ["opioid use disorder severity proxy", "nonmedical use of prescription opioids", "heroin use", "substance use disorder"],
    "risk_note": [
        "A historical measure may not correspond to DSM-5 severity; verify its definition.",
        "Misuse and nonmedical use can use different survey definitions; compare instruments.",
        "Check whether the measured substance and exposure window changed.",
        "Outdated language; a wording update does not prove diagnostic equivalence."
    ]
})
# Add or revise a row here. Example:
# crosswalk.loc[len(crosswalk)] = ["older term", "candidate current term", "why the mapping needs review"]
display(crosswalk)
print("Potential wording to review:")
for term in crosswalk["legacy_term"]:
    hits = stigma_audit(term)
    if hits:
        print(f"  {term}: {hits}")

In [ ]:
response_crosswalk = """[Explain the row you edited or added and what source documentation you would check.]"""
print(response_crosswalk)

## 5. Foundational pathway: short memo

Write **150–250 words** describing one terminology mismatch and how it could change the interpretation of addiction surveillance data. Name the source definition you would need to check. Complete this section even if you also try the applied extension.

In [ ]:
memo_foundational = """[Replace this text with your 150–250 word memo.]"""
print("Memo words:", len(memo_foundational.split()))
print(memo_foundational)

## 6. Applied pathway: nearest notes (optional extension)

Change or add one search phrase in `reference_terms`, run the cell, and inspect the three closest notes. The **same fitted method** used above compares phrases with notes. A high similarity score can still be misleading: read the actual note before deciding whether the match is sensible.

In [ ]:
reference_terms = [
    "person with opioid use disorder seeking treatment",
    "transportation barrier to medication appointment",
    "fentanyl exposure and overdose risk",
]
if model is not None:
    reference_X = model.encode([normalize_text(term) for term in reference_terms])
else:
    reference_X = vectorizer.transform([normalize_text(term) for term in reference_terms])
reference_scores = cosine_similarity(reference_X, X)
nearest_rows = []
for term, scores in zip(reference_terms, reference_scores):
    for position in np.argsort(scores)[::-1][:3]:
        nearest_rows.append({"search_phrase": term,
                             "note_id": notes.iloc[position]["note_id"],
                             "score": round(float(scores[position]), 3),
                             "note_text": notes.iloc[position]["text"]})
nearest_notes = pd.DataFrame(nearest_rows)
display(nearest_notes)

In [ ]:
response_applied = """[Optional: Which search phrase did you edit? Did its nearest notes make sense? Explain one mismatch or limitation.]"""
print(response_applied)

## 7. Ethical reflection

Answer all three questions: (1) Could a workflow using real notes expose sensitive information? (2) Which terms or assumptions might encode stigma? (3) What documentation would another researcher need to reuse your crosswalk?

In [ ]:
response_ethics = """1. [Your answer]
2. [Your answer]
3. [Your answer]"""
print(response_ethics)

## 8. Export your work

Run this cell **after** completing the response cells. It saves your responses, the edited crosswalk, similarity results, a README, and metadata into a ZIP file. Colab files in `/content` can disappear when a runtime ends, so download the ZIP and your notebook before closing Colab.

In [ ]:
responses = {
    "language_and_context": response_language,
    "similarity_interpretation": response_similarity,
    "crosswalk_explanation": response_crosswalk,
    "foundational_memo": memo_foundational,
    "applied_extension": response_applied,
    "ethical_reflection": response_ethics,
}
placeholders = {key: value for key, value in responses.items() if key != "applied_extension" and "[" in value and "]" in value}
if placeholders:
    print("Before submitting, replace the bracketed placeholders in:", ", ".join(placeholders))

crosswalk.to_csv(OUTPUT_DIR / "terminology_crosswalk.csv", index=False)
sim_df.to_csv(OUTPUT_DIR / "note_similarity.csv")
nearest_notes.to_csv(OUTPUT_DIR / "nearest_notes.csv", index=False)
with (OUTPUT_DIR / "responses.md").open("w", encoding="utf-8") as f:
    for label, answer in responses.items():
        f.write(f"## {label.replace('_', ' ').title()}\n\n{answer.strip()}\n\n")
metadata = {
    "module": MODULE_ID,
    "notebook": NOTEBOOK_TITLE,
    "data_source": data_source,
    "method": method,
    "contains_phi": False,  # Change and review this field if using any other approved data.
    "notes": "Crosswalk mappings and similarity results require human review."
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(OUTPUT_DIR / "README.md").write_text(
    f"# Module 4 submission\n\nData source: {data_source}\n\nMethod: {method}\n\n"
    "Files: responses.md (written answers); terminology_crosswalk.csv (edited mappings); "
    "note_similarity.csv (pair scores); nearest_notes.csv (optional search matches); "
    "metadata.json (provenance). Review all outputs before sharing.\n",
    encoding="utf-8")
archive = shutil.make_archive("WISER_Module_4_submission", "zip", root_dir=".", base_dir="outputs")
print("Saved:", archive)
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("If running outside Colab, download the ZIP from the file browser.")